# 03 — Churn Feature Engineering

Engineers ~60 churn features for **all eligible clients** (train + test + holdout).  
All aggregations are computed over the **feature window** `[T0, feature_window_end]` only.  
Outcome-window data is strictly forbidden here (no data leakage).

**Outputs**
- `data/processed/features_churn.parquet` — one row per eligible client, ~60 feature columns
- `data/models/churn_feature_columns.json` — ordered list of feature column names
- `data/models/churn_category_whitelist.json` — top 15 merchant categories by outflow

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import json
import re

import numpy as np
import pandas as pd

import config

## 1. Helpers

In [3]:
def parse_datetime(s):
    """Parse timestamp columns with trailing slash: 'd/m/yyyy/ HH:MM:SS'."""
    s = s.astype(str).str.strip().str.replace(r'(\d{4})/', r'\1', regex=True)
    return pd.to_datetime(s, format='%d/%m/%Y %H:%M:%S', errors='coerce', utc=True)


def parse_date_only(s):
    """Parse date-only columns with trailing slash: 'd/m/yyyy/'."""
    s = s.astype(str).str.strip().str.replace(r'(\d{4})/', r'\1', regex=True)
    return pd.to_datetime(s, format='%d/%m/%Y', errors='coerce', utc=True)


def read_transactions(path):
    """Read TRANSAKCIJE CSV handling the extra phantom semicolon column."""
    cols = [
        'IDENTIFIKATOR_PROIZVODA',
        'DATUM_I_VRIJEME_TRANSAKCIJE',
        'IZNOS_TRANSAKCIJE_U_DOMICILNOJ_VALUTI',
        'VALUTA_TRANSAKCIJE',
        'KANAL',
        'SMJER',
        'DRZAVA_DRUGE_STRANE',
        'DJELATNOST_DRUGE_STRANE',
        'KATEGORIJA_DJELATNOSTI_DRUGE_STRANE',
        'VRSTA_TRANSAKCIJE',
    ]
    df = pd.read_csv(
        path,
        sep=';',
        encoding='latin-1',
        names=cols + ['EXTRA'],
        header=0,
        low_memory=False,
    )
    mask = df['EXTRA'].notna()
    # Assumption: when EXTRA is non-null the CSV parser saw one extra
    # field because the source file has a trailing semicolon on every
    # line, causing a one-position right-shift for the last two columns.
    # The repair above rejoins the split KATEGORIJA value and moves
    # VRSTA_TRANSAKCIJE back to its correct column.
    # Sanity-check: rows WITHOUT the shift must have no stray EXTRA value.
    assert df.loc[~mask, 'EXTRA'].isna().all() or (
        df.loc[~mask, 'EXTRA'].astype(str).str.strip() == ''
    ).all(), 'Unexpected EXTRA values in non-shifted rows — CSV format may have changed'
    print(f'Phantom-column rows repaired: {mask.sum():,}')
    df.loc[mask, 'KATEGORIJA_DJELATNOSTI_DRUGE_STRANE'] = (
        df.loc[mask, 'KATEGORIJA_DJELATNOSTI_DRUGE_STRANE'].astype(str)
        + ';'
        + df.loc[mask, 'VRSTA_TRANSAKCIJE'].astype(str)
    )
    df.loc[mask, 'VRSTA_TRANSAKCIJE'] = df.loc[mask, 'EXTRA']
    return df.drop(columns=['EXTRA'])


def safe_div(num, denom):
    """Element-wise safe division. Zero or NaN denominator returns NaN."""
    num = pd.to_numeric(num, errors='coerce')
    denom = pd.to_numeric(denom, errors='coerce')
    result = num / denom
    result[denom == 0] = np.nan
    return result


def ols_slope(series):
    """OLS slope of a Series over its integer index positions. Returns NaN if fewer than 2 points."""
    vals = series.dropna()
    n = len(vals)
    if n < 2:
        return np.nan
    x = np.arange(n, dtype=float)
    x_bar = x.mean()
    y_bar = vals.mean()
    num = ((x - x_bar) * (vals.values - y_bar)).sum()
    den = ((x - x_bar) ** 2).sum()
    return num / den if den != 0 else np.nan


def normalize_ascii(text):
    """Normalize Croatian/Bosnian diacritics to ASCII via manual replacement."""
    if not isinstance(text, str):
        return ''
    replacements = {
        'c\u030c': 'c', '\u010d': 'c', '\u0107': 'c',
        's\u030c': 's', '\u0161': 's',
        'z\u030c': 'z', '\u017e': 'z',
        '\u0111': 'd',
        'C\u030c': 'C', '\u010c': 'C', '\u0106': 'C',
        'S\u030c': 'S', '\u0160': 'S',
        'Z\u030c': 'Z', '\u017d': 'Z',
        '\u0110': 'D',
        # Latin-1 sequences that appear in data read as latin-1
        '\xe8': 'c', '\xe6': 'c', '\xf0': 'd', '\xfe': 'z',
        '\xc8': 'C', '\xc6': 'C', '\xd0': 'D', '\xde': 'Z',
    }
    for src, dst in replacements.items():
        text = text.replace(src, dst)
    return text.encode('ascii', errors='ignore').decode('ascii')


def sanitize_column_name(cat_str):
    """Convert category string to UPPER_SNAKE_CASE ASCII column name fragment."""
    s = normalize_ascii(cat_str).upper()
    s = re.sub(r'[^A-Z0-9]+', '_', s)
    s = re.sub(r'_+', '_', s)
    return s.strip('_')

## 2. Load raw tables and time windows

In [4]:
RAW    = config.RAW_DATA_DIR
PROC   = config.PROCESSED_DATA_DIR
MODELS = config.MODELS_DIR
MODELS.mkdir(parents=True, exist_ok=True)

with open(PROC / 'time_split.json') as f:
    ts = json.load(f)

T0                 = pd.Timestamp(ts['T0'])
T_end              = pd.Timestamp(ts['T_end'])
feature_window_end = pd.Timestamp(ts['feature_window_end'])

print(f'T0                 : {T0}')
print(f'feature_window_end : {feature_window_end}')
print(f'T_end              : {T_end}')
print(f'Outcome window days: {(T_end - feature_window_end).days}')

T0                 : 2024-06-02 23:25:39+00:00
feature_window_end : 2025-12-15 23:55:04+00:00
T_end              : 2026-04-14 23:55:04+00:00
Outcome window days: 120


In [5]:
with open(PROC / 'split_train.json')   as f: train_ids   = json.load(f)
with open(PROC / 'split_test.json')    as f: test_ids    = json.load(f)
with open(PROC / 'split_holdout.json') as f: holdout_ids = json.load(f)

all_eligible_ids = set(train_ids) | set(test_ids) | set(holdout_ids)
print(f'Train: {len(train_ids):,}  Test: {len(test_ids):,}  Holdout: {len(holdout_ids):,}  Total: {len(all_eligible_ids):,}')

Train: 3,126  Test: 1,026  Holdout: 462  Total: 4,614


In [6]:
clients = pd.read_parquet(PROC / 'clients_eligible.parquet')
print(f'clients_eligible shape: {clients.shape}')
assert set(clients['IDENTIFIKATOR_KLIJENTA']) == all_eligible_ids, 'Client universe mismatch!'

clients_eligible shape: (4614, 39)


In [7]:
products = pd.read_csv(RAW / 'HACKATHON ZICER 202604 PROIZVODI.csv', sep=';', encoding='latin-1', low_memory=False)
for col in ['DATUM_OTVARANJA', 'DATUM_ZATVARANJA']:
    products[col] = parse_date_only(products[col])
print(f'Products raw shape: {products.shape}')

# Restrict to eligible clients — speeds up downstream joins
products = products[products['IDENTIFIKATOR_KLIJENTA'].isin(all_eligible_ids)].copy()
print(f'Products for eligible clients: {products.shape}')

Products raw shape: (58703, 13)
Products for eligible clients: (39031, 13)


In [8]:
tx_raw = read_transactions(RAW / 'HACKATHON ZICER 202604 TRANSAKCIJE.csv')
tx_raw['DATUM_I_VRIJEME_TRANSAKCIJE'] = parse_datetime(tx_raw['DATUM_I_VRIJEME_TRANSAKCIJE'])
tx_raw['IZNOS_TRANSAKCIJE_U_DOMICILNOJ_VALUTI'] = (
    tx_raw['IZNOS_TRANSAKCIJE_U_DOMICILNOJ_VALUTI']
    .astype(str).str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)
print(f'Transactions raw shape: {tx_raw.shape}')

Phantom-column rows repaired: 53,267
Transactions raw shape: (1187661, 10)


In [9]:
balances_raw = pd.read_csv(
    RAW / 'HACKATHON ZICER 202604 STANJA PROIZVODA.csv',
    sep=';', encoding='latin-1', low_memory=False
)
balances_raw['STANJE_U_DOMICILNOJ_VALUTI'] = (
    balances_raw['STANJE_U_DOMICILNOJ_VALUTI']
    .astype(str).str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)
balances_raw['VRIJEDI_OD'] = parse_date_only(balances_raw['VRIJEDI_OD'])
balances_raw['VRIJEDI_DO'] = parse_date_only(balances_raw['VRIJEDI_DO'])
print(f'Balances raw shape: {balances_raw.shape}')

Balances raw shape: (817933, 6)


In [10]:
# Shared product → client lookup
prod_to_client = (
    products[['IDENTIFIKATOR_PROIZVODA', 'IDENTIFIKATOR_KLIJENTA']]
    .drop_duplicates()
)

# Transactions: join to eligible clients, STRICT feature-window filter
tx = tx_raw.merge(prod_to_client, on='IDENTIFIKATOR_PROIZVODA', how='left')
tx = tx[tx['IDENTIFIKATOR_KLIJENTA'].isin(all_eligible_ids)].copy()
tx = tx[
    tx['DATUM_I_VRIJEME_TRANSAKCIJE'].notna()
    & (tx['DATUM_I_VRIJEME_TRANSAKCIJE'] >= T0)
    & (tx['DATUM_I_VRIJEME_TRANSAKCIJE'] <= feature_window_end)
].copy()

max_tx_date = tx['DATUM_I_VRIJEME_TRANSAKCIJE'].max()
print(f'Transactions in feature window: {len(tx):,}')
print(f'Max tx date: {max_tx_date}  (must be <= {feature_window_end})')
assert max_tx_date <= feature_window_end, 'LEAKAGE: tx date exceeds feature_window_end!'

# Balances: join to eligible clients
balances = balances_raw.merge(prod_to_client, on='IDENTIFIKATOR_PROIZVODA', how='left')
balances = balances[balances['IDENTIFIKATOR_KLIJENTA'].isin(all_eligible_ids)].copy()
print(f'Balance rows for eligible clients: {len(balances):,}')

Transactions in feature window: 934,963
Max tx date: 2025-12-15 23:51:58+00:00  (must be <= 2025-12-15 23:55:04+00:00)
Balance rows for eligible clients: 759,894


## 3. Group 1 — Demographics & employment

Source: `KLIJENTI` (via `clients_eligible.parquet`).  
Categorical columns kept as strings — LightGBM handles them natively.  
DOB → `age` computed at `feature_window_end`.

In [11]:
demo = clients[['IDENTIFIKATOR_KLIJENTA']].copy()

dob = parse_date_only(clients['DOB'])
demo['age'] = (feature_window_end - dob).dt.days / 365.25

demo['gender']                          = clients['SPOL'].astype(str)
demo['credit_rating']                   = pd.to_numeric(clients['KREDITNI_RATING'], errors='coerce')
demo['employment_status']               = clients['STATUS_ZAPOSLENJA'].astype(str)
demo['employment_type']                 = clients['VRSTA_ZAPOSLENJA'].astype(str)
demo['position']                        = clients['POZICIJA'].astype(str)
demo['education']                       = clients['STRUCNA_SPREMA'].astype(str)
demo['occupation']                      = clients['ZANIMANJE'].astype(str)
demo['employer_category']               = clients['KATEGORIJA_POSLODAVCA'].astype(str)
demo['employer_type']                   = clients['TIP_POSLODAVCA'].astype(str)
demo['years_working_total']             = pd.to_numeric(clients['GODINE_STAZA'], errors='coerce')
demo['years_at_current_employer']       = pd.to_numeric(clients['GODINE_STAZA_KOD_TRENUTNOG_POSLODAVCA'], errors='coerce')
demo['household_size']                  = pd.to_numeric(clients['BROJ_CLANOVA_KUCANSTVA'], errors='coerce')
demo['dependents']                      = pd.to_numeric(clients['BROJ_UZDRZAVANIH_CLANOVA_KUCANSTVA'], errors='coerce')
demo['housing_type']                    = clients['VRSTA_STANOVANJA'].astype(str)
demo['ownership_type']                  = clients['TIP_VLASNISTVA'].astype(str)
demo['marital_status']                  = clients['BRACNI_STATUS'].astype(str)
demo['is_resident']                     = clients['KLIJENT_REZIDENT'].astype(str).str.strip().str.upper().isin(['1', 'Y', 'D', 'DA', 'TRUE', 'T']).astype(int)
demo['receives_primary_income_at_bank'] = clients['KLIJENT_PRIMA_OSNOVNO_PRIMANJE_U_BANCI'].astype(str).str.strip().str.upper().isin(['1', 'Y', 'D', 'DA', 'TRUE', 'T']).astype(int)

print(f'Demographics frame: {demo.shape}')
demo.head(2)

Demographics frame: (4614, 20)


,IDENTIFIKATOR_KLIJENTA,age,gender,credit_rating,employment_status,employment_type,position,education,occupation,employer_category,employer_type,years_working_total,years_at_current_employer,household_size,dependents,housing_type,ownership_type,marital_status,is_resident,receives_primary_income_at_bank
0,53NHZ40D7ZK3,NaN,F,NaN,EMPLOYED,OSTALO,OSTALO,None,None,None,None,NaN,NaN,NaN,NaN,None,None,None,1,0
1,JCFRFWOVBXZY,NaN,F,NaN,EMPLOYED,ZAPOSLEN,ZAPOSLENIK,SSS,OSTALI ZAPOSLENICI,None,PRIVATNO PODUZEÆE - MALO,NaN,NaN,NaN,NaN,None,None,None,1,0


## 4. Group 2 — Tenure & acquisition

In [12]:
tenure = clients[['IDENTIFIKATOR_KLIJENTA']].copy()

first_rel = parse_date_only(clients['DATUM_PRVOG_POCETKA_POSLOVNOG_ODNOSA'])
last_rel  = parse_date_only(clients['DATUM_ZADNJEG_POCETKA_POSLOVNOG_ODNOSA'])

tenure['months_since_first_relationship']      = (feature_window_end - first_rel).dt.days / 30.4375
tenure['months_since_last_relationship_start'] = (feature_window_end - last_rel).dt.days / 30.4375
tenure['acquisition_channel']                  = clients['OZNAKA_AKVIZICIJE_KLIJENTA'].astype(str)

print(f'Tenure frame: {tenure.shape}')
tenure.head(2)

Tenure frame: (4614, 4)


,IDENTIFIKATOR_KLIJENTA,months_since_first_relationship,months_since_last_relationship_start,acquisition_channel
0,53NHZ40D7ZK3,NaN,NaN,HPB
1,JCFRFWOVBXZY,NaN,NaN,HPB


## 5. Group 3 — Product holdings

Products opened on or before `feature_window_end`.  
"Active" = not closed, or closed after `feature_window_end`.  
"Closed last 12m" = `DATUM_ZATVARANJA` in `[feature_window_end - 365d, feature_window_end]`.

### Product flag heuristics

Conservative case-insensitive substring match on the concatenated normalized text of  
`NAZIV_DOMENE_PROIZVODA` / `NAZIV_KATEGORIJE_PROIZVODA` / `NAZIV_KLASE_PROIZVODA`.

| Flag | Include terms | Exclude terms |
|------|--------------|---------------|
| `has_checking` | TEKUC, ZIRO, CURRENT, CHECKING | — |
| `has_savings` | STEDN, DEPOZIT, SAVING, DEPOSIT | INVEST |
| `has_credit_card` | (KARTIC or CARD) AND (KREDIT or CREDIT); or card brands with KREDIT | — |
| `has_loan` | KREDIT, LOAN | KARTIC, CARD, HIPOT, MORTGAGE |
| `has_mortgage` | HIPOT, STAMBEN, MORTGAGE | — |
| `has_investment` | INVEST, FOND, FUND, VRIJEDNOSN, DIONIC | — |

In [13]:
# Product flag rules — single dict for easy review/adjustment
FLAG_RULES = {
    'has_checking': {'include': ['TEKUC', 'ZIRO', 'CURRENT', 'CHECKING'], 'exclude': []},
    'has_savings':  {'include': ['STEDN', 'DEPOZIT', 'SAVING', 'DEPOSIT'], 'exclude': ['INVEST']},
    'has_credit_card': None,  # handled separately
    'has_loan':     {'include': ['KREDIT', 'LOAN'], 'exclude': ['KARTIC', 'CARD', 'HIPOT', 'MORTGAGE']},
    'has_mortgage': {'include': ['HIPOT', 'STAMBEN', 'MORTGAGE'], 'exclude': []},
    'has_investment': {'include': ['INVEST', 'FOND', 'FUND', 'VRIJEDNOSN', 'DIONIC'], 'exclude': []},
}

prod_fw = products[
    products['DATUM_OTVARANJA'].notna()
    & (products['DATUM_OTVARANJA'] <= feature_window_end)
].copy()

for col in ['NAZIV_DOMENE_PROIZVODA', 'NAZIV_KATEGORIJE_PROIZVODA', 'NAZIV_KLASE_PROIZVODA']:
    prod_fw[col + '_NORM'] = prod_fw[col].fillna('').apply(normalize_ascii).str.upper()

prod_fw['_TEXT'] = (
    prod_fw['NAZIV_DOMENE_PROIZVODA_NORM'] + ' '
    + prod_fw['NAZIV_KATEGORIJE_PROIZVODA_NORM'] + ' '
    + prod_fw['NAZIV_KLASE_PROIZVODA_NORM']
)

def matches_rule(text, rules):
    if rules is None:
        return False
    return (any(t in text for t in rules['include'])
            and not any(t in text for t in rules['exclude']))

for flag, rules in FLAG_RULES.items():
    if rules is not None:
        prod_fw[flag] = prod_fw['_TEXT'].apply(lambda t: matches_rule(t, rules)).astype(int)

prod_fw['has_credit_card'] = prod_fw['_TEXT'].apply(
    lambda t: (
        (('KARTIC' in t) or ('CARD' in t)) and (('KREDIT' in t) or ('CREDIT' in t))
    ) or (
        any(b in t for b in ['VISA', 'MASTERCARD', 'MAESTRO']) and ('KREDIT' in t)
    )
).astype(int)

print('Product flag distribution (per product row):')
for flag in ['has_checking', 'has_savings', 'has_credit_card', 'has_loan', 'has_mortgage', 'has_investment']:
    print(f'  {flag}: {prod_fw[flag].sum():,}')

Product flag distribution (per product row):
  has_checking: 6,461
  has_savings: 3,484
  has_credit_card: 3,969
  has_loan: 2,314
  has_mortgage: 162
  has_investment: 3


In [14]:
prod_fw['is_active'] = (
    prod_fw['DATUM_ZATVARANJA'].isna()
    | (prod_fw['DATUM_ZATVARANJA'] > feature_window_end)
).astype(int)

closed_12m_start = feature_window_end - pd.Timedelta(days=365)
prod_fw['closed_last_12m'] = (
    prod_fw['DATUM_ZATVARANJA'].notna()
    & (prod_fw['DATUM_ZATVARANJA'] >= closed_12m_start)
    & (prod_fw['DATUM_ZATVARANJA'] <= feature_window_end)
).astype(int)

flag_cols = ['has_checking', 'has_savings', 'has_credit_card', 'has_loan', 'has_mortgage', 'has_investment']

prod_agg = prod_fw.groupby('IDENTIFIKATOR_KLIJENTA').agg(
    n_products_total=('IDENTIFIKATOR_PROIZVODA', 'count'),
    n_products_active=('is_active', 'sum'),
    n_products_closed_last_12m=('closed_last_12m', 'sum'),
    n_domains=('NAZIV_DOMENE_PROIZVODA', 'nunique'),
    n_categories=('NAZIV_KATEGORIJE_PROIZVODA', 'nunique'),
    **{flag: (flag, 'max') for flag in flag_cols},
).reset_index()

prod_agg['product_diversity_score'] = prod_agg['n_domains'] + prod_agg['n_categories'] / 10.0

print(f'Product aggregation shape: {prod_agg.shape}')
prod_agg.head(2)

Product aggregation shape: (4614, 13)


,IDENTIFIKATOR_KLIJENTA,n_products_total,n_products_active,n_products_closed_last_12m,n_domains,n_categories,has_checking,has_savings,has_credit_card,has_loan,has_mortgage,has_investment,product_diversity_score
0,00J3RILV5U6Z,2,1,1,2,2,1,1,0,0,0,0,2.2
1,00KLLUYR5RVU,17,4,1,4,9,1,0,1,1,0,0,4.9


## 6. Group 4 — Balance features

Source: `STANJA_PROIZVODA` clipped to `[T0, feature_window_end]`.  
Strategy: build a monthly total-balance series per client, then derive aggregates.

In [15]:
# SCD overlap filter: include any balance record whose validity interval
# overlaps [T0, feature_window_end].
# A record overlaps the window when:
#   VRIJEDI_OD <= feature_window_end  (started before the window closed)
#   AND VRIJEDI_DO IS NULL OR >= T0   (still valid at or after window start)
# The original VRIJEDI_OD >= T0 guard incorrectly dropped records that
# were active during the window but started before T0.
bal_fw = balances[
    balances['VRIJEDI_OD'].notna()
    & (balances['VRIJEDI_OD'] <= feature_window_end)
    & (balances['VRIJEDI_DO'].isna() | (balances['VRIJEDI_DO'] >= T0))
].copy()

print(f'Balance rows in feature window: {len(bal_fw):,}')

# Clip VRIJEDI_OD to T0 so that pre-T0 start dates are not used as
# month bucket keys — every bucket must fall inside [T0, feature_window_end].
bal_fw['VRIJEDI_OD'] = bal_fw['VRIJEDI_OD'].clip(lower=T0)

# Assign month bucket, then sum across products per client per month
bal_fw['month'] = (
    bal_fw['VRIJEDI_OD'].dt.to_period('M')
    .dt.to_timestamp(how='start')
    .dt.tz_localize('UTC')
)

monthly_bal = (
    bal_fw.groupby(['IDENTIFIKATOR_KLIJENTA', 'month'])['STANJE_U_DOMICILNOJ_VALUTI']
    .sum()
    .reset_index()
    .rename(columns={'STANJE_U_DOMICILNOJ_VALUTI': 'total_balance'})
)
print(f'Monthly balance rows: {len(monthly_bal):,}')

Balance rows in feature window: 634,434
Monthly balance rows: 56,857


/var/folders/zn/kwwpw8d5773232__xbr6ncdr0000gp/T/ipykernel_68254/2109712652.py:22: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  bal_fw['VRIJEDI_OD'].dt.to_period('M')


In [16]:
def compute_balance_features(group_df):
    bal = group_df.sort_values('month')['total_balance']
    if len(bal) == 0:
        return pd.Series({
            'total_balance_avg': np.nan, 'total_balance_end': np.nan,
            'total_balance_start': np.nan, 'balance_trend_slope': np.nan,
            'balance_volatility': np.nan, 'balance_min': np.nan,
            'balance_max': np.nan, 'balance_end_to_start_ratio': np.nan,
        })
    start_val = bal.iloc[0]
    end_val   = bal.iloc[-1]
    return pd.Series({
        'total_balance_avg':          bal.mean(),
        'total_balance_end':          end_val,
        'total_balance_start':        start_val,
        'balance_trend_slope':        ols_slope(bal),
        'balance_volatility':         bal.std(),
        'balance_min':                bal.min(),
        'balance_max':                bal.max(),
        'balance_end_to_start_ratio': (
            end_val / start_val
            if (not pd.isna(start_val)) and (start_val > 0)
            else np.nan
        ),
    })

bal_features = (
    monthly_bal.groupby('IDENTIFIKATOR_KLIJENTA')
    .apply(compute_balance_features)
    .reset_index()
)

print(f'Balance features shape: {bal_features.shape}')
bal_features.head(2)

Balance features shape: (3281, 9)


/var/folders/zn/kwwpw8d5773232__xbr6ncdr0000gp/T/ipykernel_68254/3457664757.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_bal.groupby('IDENTIFIKATOR_KLIJENTA')


,IDENTIFIKATOR_KLIJENTA,total_balance_avg,total_balance_end,total_balance_start,balance_trend_slope,balance_volatility,balance_min,balance_max,balance_end_to_start_ratio
0,00J3RILV5U6Z,34637.755789,296.32,49994.52,-4815.259825,47707.411063,296.30,152631.94,0.005927
1,00KLLUYR5RVU,8742.985263,6008.47,5717.15,-1.574298,6129.115613,1017.05,26787.52,1.050955


## 7. Group 5 — Transaction features

Source: `TRANSAKCIJE`, feature window only.

### SMJER (direction) mapping

Data inspection reveals two values:
- `D` — *Dugovna* (debit = **outflow**)
- `C` — *Creditna* (credit = **inflow**)

In [17]:
INFLOW_VALUES  = {'C'}  # Credit / inflow
OUTFLOW_VALUES = {'D'}  # Debit  / outflow

tx['is_inflow']  = tx['SMJER'].isin(INFLOW_VALUES)
tx['is_outflow'] = tx['SMJER'].isin(OUTFLOW_VALUES)
tx['abs_amount'] = tx['IZNOS_TRANSAKCIJE_U_DOMICILNOJ_VALUTI'].abs()

print('SMJER value counts in feature window:')
print(tx['SMJER'].value_counts().to_dict())

SMJER value counts in feature window:
{'D': 780749, 'C': 154214}


In [18]:
cutoff_30d  = feature_window_end - pd.Timedelta(days=30)
cutoff_90d  = feature_window_end - pd.Timedelta(days=90)
cutoff_180d = feature_window_end - pd.Timedelta(days=180)

tx['in_last_30d']  = tx['DATUM_I_VRIJEME_TRANSAKCIJE'] >= cutoff_30d
tx['in_last_90d']  = tx['DATUM_I_VRIJEME_TRANSAKCIJE'] >= cutoff_90d
tx['in_last_180d'] = tx['DATUM_I_VRIJEME_TRANSAKCIJE'] >= cutoff_180d
tx['month'] = (
    tx['DATUM_I_VRIJEME_TRANSAKCIJE'].dt.to_period('M')
    .dt.to_timestamp(how='start')
    .dt.tz_localize('UTC')
)

print(f'Last-30d cutoff : {cutoff_30d}')
print(f'Last-90d cutoff : {cutoff_90d}')
print(f'Last-180d cutoff: {cutoff_180d}')

Last-30d cutoff : 2025-11-15 23:55:04+00:00
Last-90d cutoff : 2025-09-16 23:55:04+00:00
Last-180d cutoff: 2025-06-18 23:55:04+00:00


/var/folders/zn/kwwpw8d5773232__xbr6ncdr0000gp/T/ipykernel_68254/2953423010.py:9: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  tx['DATUM_I_VRIJEME_TRANSAKCIJE'].dt.to_period('M')


In [19]:
# Pre-compute per-row boolean flags as int for clean groupby agg
tx['_inflow_amt']  = tx['IZNOS_TRANSAKCIJE_U_DOMICILNOJ_VALUTI'].where(tx['is_inflow'], 0)
tx['_outflow_amt'] = tx['abs_amount'].where(tx['is_outflow'], 0)
tx['_amt_90d']     = tx['abs_amount'].where(tx['in_last_90d'], 0)

tx_base = tx.groupby('IDENTIFIKATOR_KLIJENTA').agg(
    tx_count_total     = ('IDENTIFIKATOR_PROIZVODA', 'count'),
    tx_count_last90d   = ('in_last_90d',  'sum'),
    tx_count_last30d   = ('in_last_30d',  'sum'),
    tx_count_last180d  = ('in_last_180d', 'sum'),
    tx_amount_total    = ('abs_amount',    'sum'),
    tx_amount_last90d  = ('_amt_90d',      'sum'),
    tx_inflow_total    = ('_inflow_amt',   'sum'),
    tx_outflow_total   = ('_outflow_amt',  'sum'),
    _last_tx_ts        = ('DATUM_I_VRIJEME_TRANSAKCIJE', 'max'),
).reset_index()

tx_base['days_since_last_tx'] = (
    (feature_window_end - tx_base['_last_tx_ts'])
    .dt.days
    .where(tx_base['_last_tx_ts'].notna(), np.nan)
)

tx_median = (
    tx.groupby('IDENTIFIKATOR_KLIJENTA')['abs_amount']
    .median()
    .rename('tx_amount_median')
    .reset_index()
)
tx_base = tx_base.merge(tx_median, on='IDENTIFIKATOR_KLIJENTA', how='left')

tx_base['tx_amount_avg_per_tx']    = safe_div(tx_base['tx_amount_total'], tx_base['tx_count_total'])
tx_base['tx_net_flow']             = tx_base['tx_inflow_total'] - tx_base['tx_outflow_total']
tx_base['tx_count_ratio_30d_180d'] = safe_div(tx_base['tx_count_last30d'], tx_base['tx_count_last180d'])

print(f'tx_base shape: {tx_base.shape}')
tx_base.head(2)

tx_base shape: (4614, 15)


,IDENTIFIKATOR_KLIJENTA,tx_count_total,tx_count_last90d,tx_count_last30d,tx_count_last180d,tx_amount_total,tx_amount_last90d,tx_inflow_total,tx_outflow_total,_last_tx_ts,days_since_last_tx,tx_amount_median,tx_amount_avg_per_tx,tx_net_flow,tx_count_ratio_30d_180d
0,00J3RILV5U6Z,17,1,0,2,154972.61,0.01,52641.14,102331.47,2025-10-31 23:49:50+00:00,45,0.020,9116.035882,-49690.33,0.000000
1,00KLLUYR5RVU,444,82,24,162,56707.79,9721.40,28204.36,28503.43,2025-12-15 23:47:06+00:00,0,26.305,127.720248,-299.07,0.148148


In [20]:
monthly_tx = (
    tx.groupby(['IDENTIFIKATOR_KLIJENTA', 'month'])
    .agg(month_count=('IDENTIFIKATOR_PROIZVODA', 'count'),
         month_amount=('abs_amount', 'sum'))
    .reset_index()
)

def compute_tx_trends(g):
    g = g.sort_values('month')
    avg_monthly = g['month_count'].mean()
    last_month  = g['month_count'].iloc[-1] if len(g) > 0 else np.nan
    return pd.Series({
        'tx_count_trend_slope':  ols_slope(g['month_count']),
        'tx_amount_trend_slope': ols_slope(g['month_amount']),
        'tx_month_count_ratio':  last_month / avg_monthly if avg_monthly > 0 else np.nan,
    })

tx_trends = (
    monthly_tx.groupby('IDENTIFIKATOR_KLIJENTA')
    .apply(compute_tx_trends)
    .reset_index()
)

tx_features = (
    tx_base
    .merge(tx_trends, on='IDENTIFIKATOR_KLIJENTA', how='left')
    .drop(columns=['tx_count_last180d', '_last_tx_ts'])
)

print(f'tx_features shape: {tx_features.shape}')

tx_features shape: (4614, 16)


/var/folders/zn/kwwpw8d5773232__xbr6ncdr0000gp/T/ipykernel_68254/303830535.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_tx.groupby('IDENTIFIKATOR_KLIJENTA')


## 8. Group 6 — Channel features

### KANAL mapping

| Raw KANAL value | Bucket |
|---|---|
| `KARTIČNA TRANSAKCIJA` | `pos` (point-of-sale / card terminal) |
| `BANK` | `branch` (teller/branch transaction) |
| `Retail internet banking` | `online` |
| `Croatian Post` | `branch` (physical payment point) |
| `Standing instructions` | `online` (automated digital instruction) |
| `SEPA Instant` | `online` (electronic payment) |
| `Croatian Post payment order files` | `branch` |

`mobile` and `atm` buckets: no explicit mobile or ATM channel present in data — will be 0.

In [21]:
CHANNEL_MAP = {
    'KARTICNA TRANSAKCIJA':               'pos',
    'BANK':                               'branch',
    'RETAIL INTERNET BANKING':            'online',
    'CROATIAN POST':                      'branch',
    'STANDING INSTRUCTIONS':              'online',
    'SEPA INSTANT':                       'online',
    'CROATIAN POST PAYMENT ORDER FILES':  'branch',
}
CHANNEL_BUCKETS = ['atm', 'online', 'branch', 'mobile', 'pos']

def map_channel(raw):
    if pd.isna(raw):
        return 'other'
    return CHANNEL_MAP.get(normalize_ascii(str(raw)).upper(), 'other')

tx['channel_bucket'] = tx['KANAL'].apply(map_channel)

print('Channel bucket distribution:')
print(tx['channel_bucket'].value_counts().to_dict())

Channel bucket distribution:
{'pos': 579009, 'branch': 275943, 'online': 80011}


In [22]:
ch_diversity = (
    tx.groupby('IDENTIFIKATOR_KLIJENTA')['channel_bucket']
    .nunique()
    .rename('channel_diversity')
    .reset_index()
)

tx_total_count = tx.groupby('IDENTIFIKATOR_KLIJENTA').size().rename('_total').reset_index()

ch_counts = (
    tx.groupby(['IDENTIFIKATOR_KLIJENTA', 'channel_bucket'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
for bucket in CHANNEL_BUCKETS:
    if bucket not in ch_counts.columns:
        ch_counts[bucket] = 0

ch_counts = ch_counts.merge(tx_total_count, on='IDENTIFIKATOR_KLIJENTA')
for bucket in CHANNEL_BUCKETS:
    ch_counts[f'share_by_channel_{bucket}'] = safe_div(ch_counts[bucket], ch_counts['_total'])

# Mobile adoption in last 90d
tx_last90 = tx[tx['in_last_90d']]
ch_last90_total = tx_last90.groupby('IDENTIFIKATOR_KLIJENTA').size().rename('_total_90d').reset_index()
mobile_last90 = (
    tx_last90[tx_last90['channel_bucket'] == 'mobile']
    .groupby('IDENTIFIKATOR_KLIJENTA').size()
    .rename('_mobile_90d').reset_index()
)
mobile_share = ch_last90_total.merge(mobile_last90, on='IDENTIFIKATOR_KLIJENTA', how='left')
mobile_share['_mobile_90d'] = mobile_share['_mobile_90d'].fillna(0)
mobile_share['mobile_adoption_last90d'] = safe_div(mobile_share['_mobile_90d'], mobile_share['_total_90d'])

share_cols = [f'share_by_channel_{b}' for b in CHANNEL_BUCKETS]
channel_features = (
    ch_diversity
    .merge(ch_counts[['IDENTIFIKATOR_KLIJENTA'] + share_cols], on='IDENTIFIKATOR_KLIJENTA', how='left')
    .merge(mobile_share[['IDENTIFIKATOR_KLIJENTA', 'mobile_adoption_last90d']], on='IDENTIFIKATOR_KLIJENTA', how='left')
)

print(f'Channel features shape: {channel_features.shape}')
channel_features.head(2)

Channel features shape: (4614, 8)


,IDENTIFIKATOR_KLIJENTA,channel_diversity,share_by_channel_atm,share_by_channel_online,share_by_channel_branch,share_by_channel_mobile,share_by_channel_pos,mobile_adoption_last90d
0,00J3RILV5U6Z,1,0.0,0.00000,1.000000,0.0,0.000000,0.0
1,00KLLUYR5RVU,3,0.0,0.15991,0.173423,0.0,0.666667,0.0


## 9. Group 7 — Category spend features

Uses `KATEGORIJA_DJELATNOSTI_DRUGE_STRANE` (broad category), **outflow only**, feature window.  
Steps: compute total outflow per category → take top 15 → build `spend_share_{cat}` per client.

In [23]:
tx_outflow = tx[tx['is_outflow']].copy()
tx_outflow['KATEGORIJA_DJELATNOSTI_DRUGE_STRANE'] = (
    tx_outflow['KATEGORIJA_DJELATNOSTI_DRUGE_STRANE'].fillna('').str.strip()
)

cat_totals = (
    tx_outflow
    .groupby('KATEGORIJA_DJELATNOSTI_DRUGE_STRANE')['abs_amount']
    .sum()
    .sort_values(ascending=False)
)
# Drop empty / error / unknown categories before ranking
cat_totals = cat_totals[cat_totals.index.str.len() > 2]
cat_totals = cat_totals[~cat_totals.index.str.upper().str.startswith('ERROR')]
cat_totals = cat_totals[~cat_totals.index.str.upper().str.startswith('NEPOZN')]

# Note: KATEGORIJA_DJELATNOSTI_DRUGE_STRANE contains a mix of English
# generic labels (e.g. 'RETAIL', 'GROCERIES') and Croatian NACE-like
# descriptors (e.g. 'Trgovina na malo...').  Both forms are expected and
# intentional — the column reflects the raw MCC/NACE enrichment from the
# source system which uses inconsistent language conventions.
print('Top 20 categories by total outflow:')
print(cat_totals.head(20))

Top 20 categories by total outflow:
KATEGORIJA_DJELATNOSTI_DRUGE_STRANE
Service providers                                                                                18756788.72
Retail outlets                                                                                    6918739.42
JAVNA UPRAVA I OBRANA; OBVEZNO SOCIJALNO OSIGURANJE                                               2883793.02
FINANCIJSKE DJELATNOSTI I DJELATNOSTI OSIGURANJA                                                  2679044.25
GRAÐEVINARSTVO                                                                                    2042137.21
Miscellaneous outlets                                                                             1800627.15
Clothing outlets                                                                                  1010747.85
TRGOVINA NA VELIKO I NA MALO; POPRAVAK MOTORNIH VOZILA I MOTOCIKALA                                572298.18
Utilities                                               

In [24]:
TOP_N_CATEGORIES = 15
top_categories = cat_totals.head(TOP_N_CATEGORIES).index.tolist()

whitelist = [
    {'original': cat, 'sanitized': sanitize_column_name(cat)}
    for cat in top_categories
]

print(f'Category whitelist ({len(whitelist)} entries):')
for entry in whitelist:
    print(f'  {entry["original"]}  ->  spend_share_{entry["sanitized"]}')

Category whitelist (15 entries):
  Service providers  ->  spend_share_SERVICE_PROVIDERS
  Retail outlets  ->  spend_share_RETAIL_OUTLETS
  JAVNA UPRAVA I OBRANA; OBVEZNO SOCIJALNO OSIGURANJE  ->  spend_share_JAVNA_UPRAVA_I_OBRANA_OBVEZNO_SOCIJALNO_OSIGURANJE
  FINANCIJSKE DJELATNOSTI I DJELATNOSTI OSIGURANJA  ->  spend_share_FINANCIJSKE_DJELATNOSTI_I_DJELATNOSTI_OSIGURANJA
  GRAÐEVINARSTVO  ->  spend_share_GRADEVINARSTVO
  Miscellaneous outlets  ->  spend_share_MISCELLANEOUS_OUTLETS
  Clothing outlets  ->  spend_share_CLOTHING_OUTLETS
  TRGOVINA NA VELIKO I NA MALO; POPRAVAK MOTORNIH VOZILA I MOTOCIKALA  ->  spend_share_TRGOVINA_NA_VELIKO_I_NA_MALO_POPRAVAK_MOTORNIH_VOZILA_I_MOTOCIKALA
  Utilities  ->  spend_share_UTILITIES
  PRERAÐIVAÈKA INDUSTRIJA  ->  spend_share_PRERADIVACKA_INDUSTRIJA
  INFORMACIJE I KOMUNIKACIJE  ->  spend_share_INFORMACIJE_I_KOMUNIKACIJE
  OPSKRBA ELEKTRIÈNOM ENERGIJOM, PLINOM, PAROM I KLIMATIZACIJA  ->  spend_share_OPSKRBA_ELEKTRICNOM_ENERGIJOM_PLINOM_PAROM_I_K

In [25]:
# Client total outflow (denominator)
client_total_outflow = (
    tx_outflow.groupby('IDENTIFIKATOR_KLIJENTA')['abs_amount']
    .sum()
    .rename('total_outflow')
    .reset_index()
)

# Pivot: client x whitelisted category -> outflow amount
tx_wl = tx_outflow[tx_outflow['KATEGORIJA_DJELATNOSTI_DRUGE_STRANE'].isin(top_categories)].copy()
cat_pivot = (
    tx_wl.groupby(['IDENTIFIKATOR_KLIJENTA', 'KATEGORIJA_DJELATNOSTI_DRUGE_STRANE'])['abs_amount']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)
for cat in top_categories:
    if cat not in cat_pivot.columns:
        cat_pivot[cat] = 0.0

cat_pivot = cat_pivot.merge(client_total_outflow, on='IDENTIFIKATOR_KLIJENTA', how='left')

for entry in whitelist:
    col = f'spend_share_{entry["sanitized"]}'
    cat_pivot[col] = safe_div(cat_pivot[entry['original']], cat_pivot['total_outflow']).fillna(0.0)

# Category diversity: distinct categories (all, not just whitelist)
cat_diversity = (
    tx_outflow[tx_outflow['KATEGORIJA_DJELATNOSTI_DRUGE_STRANE'].str.len() > 0]
    .groupby('IDENTIFIKATOR_KLIJENTA')['KATEGORIJA_DJELATNOSTI_DRUGE_STRANE']
    .nunique()
    .rename('category_diversity')
    .reset_index()
)

spend_share_cols = [f'spend_share_{e["sanitized"]}' for e in whitelist]
cat_features = (
    cat_pivot[['IDENTIFIKATOR_KLIJENTA'] + spend_share_cols]
    .merge(cat_diversity, on='IDENTIFIKATOR_KLIJENTA', how='left')
)

print(f'Category features shape: {cat_features.shape}')
cat_features.head(2)

Category features shape: (3800, 17)


,IDENTIFIKATOR_KLIJENTA,spend_share_SERVICE_PROVIDERS,spend_share_RETAIL_OUTLETS,spend_share_JAVNA_UPRAVA_I_OBRANA_OBVEZNO_SOCIJALNO_OSIGURANJE,spend_share_FINANCIJSKE_DJELATNOSTI_I_DJELATNOSTI_OSIGURANJA,spend_share_GRADEVINARSTVO,spend_share_MISCELLANEOUS_OUTLETS,spend_share_CLOTHING_OUTLETS,spend_share_TRGOVINA_NA_VELIKO_I_NA_MALO_POPRAVAK_MOTORNIH_VOZILA_I_MOTOCIKALA,spend_share_UTILITIES,spend_share_PRERADIVACKA_INDUSTRIJA,spend_share_INFORMACIJE_I_KOMUNIKACIJE,spend_share_OPSKRBA_ELEKTRICNOM_ENERGIJOM_PLINOM_PAROM_I_KLIMATIZACIJA,spend_share_TRANSPORTATION,spend_share_AMUSEMENT_AND_ENTERTAINMENT,spend_share_PROFESSIONAL_SERVICES_AND_MEMBERSHIP_ORGANIZATIONS,category_diversity
0,00J3RILV5U6Z,0.000000,0.00000,0.485786,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,2
1,00KLLUYR5RVU,0.524849,0.09448,0.006282,0.013413,0.0,0.011406,0.015161,0.0,0.001696,0.0,0.081496,0.004549,0.0,0.0,0.0,14


## 10. Group 8 — International & counterparty

**Domestic country code**: `HR` (Croatia) — the overwhelmingly most common value in `DRZAVA_DRUGE_STRANE`.

In [26]:
DOMESTIC_COUNTRY = 'HR'

tx['is_foreign'] = (
    tx['DRZAVA_DRUGE_STRANE'].notna()
    & (tx['DRZAVA_DRUGE_STRANE'].str.strip() != DOMESTIC_COUNTRY)
).astype(int)

intl_base = tx.groupby('IDENTIFIKATOR_KLIJENTA').agg(
    _total=('IDENTIFIKATOR_PROIZVODA', 'count'),
    _foreign=('is_foreign', 'sum'),
).reset_index()
intl_base['foreign_tx_share'] = safe_div(intl_base['_foreign'], intl_base['_total'])

counterparty_div = (
    tx[tx['DJELATNOST_DRUGE_STRANE'].notna()]
    .groupby('IDENTIFIKATOR_KLIJENTA')['DJELATNOST_DRUGE_STRANE']
    .nunique()
    .rename('counterparty_diversity')
    .reset_index()
)

intl_features = (
    intl_base[['IDENTIFIKATOR_KLIJENTA', 'foreign_tx_share']]
    .merge(counterparty_div, on='IDENTIFIKATOR_KLIJENTA', how='left')
)

print(f'International features shape: {intl_features.shape}')
intl_features.head(2)

International features shape: (4614, 3)


,IDENTIFIKATOR_KLIJENTA,foreign_tx_share,counterparty_diversity
0,00J3RILV5U6Z,0.000000,2
1,00KLLUYR5RVU,0.040541,37


## 11. Assemble full feature matrix

In [27]:
features = clients[['IDENTIFIKATOR_KLIJENTA']].copy()

for gdf in [demo, tenure, prod_agg, bal_features, tx_features, channel_features, cat_features, intl_features]:
    features = features.merge(gdf, on='IDENTIFIKATOR_KLIJENTA', how='left')

print(f'Assembled feature matrix: {features.shape}')
features.head(2)

Assembled feature matrix: (4614, 83)


,IDENTIFIKATOR_KLIJENTA,age,gender,credit_rating,employment_status,employment_type,position,education,occupation,employer_category,...,spend_share_UTILITIES,spend_share_PRERADIVACKA_INDUSTRIJA,spend_share_INFORMACIJE_I_KOMUNIKACIJE,spend_share_OPSKRBA_ELEKTRICNOM_ENERGIJOM_PLINOM_PAROM_I_KLIMATIZACIJA,spend_share_TRANSPORTATION,spend_share_AMUSEMENT_AND_ENTERTAINMENT,spend_share_PROFESSIONAL_SERVICES_AND_MEMBERSHIP_ORGANIZATIONS,category_diversity,foreign_tx_share,counterparty_diversity
0,53NHZ40D7ZK3,NaN,F,NaN,EMPLOYED,OSTALO,OSTALO,None,None,None,...,0.000454,0.0,0.0,0.0,0.0,0.0,0.000297,7.0,0.002841,20
1,JCFRFWOVBXZY,NaN,F,NaN,EMPLOYED,ZAPOSLEN,ZAPOSLENIK,SSS,OSTALI ZAPOSLENICI,None,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,5.0,0.042857,15


In [28]:
# Canonical deterministic column order
DEMO_COLS = [
    'age', 'gender', 'credit_rating', 'employment_status', 'employment_type',
    'position', 'education', 'occupation', 'employer_category', 'employer_type',
    'years_working_total', 'years_at_current_employer',
    'household_size', 'dependents', 'housing_type', 'ownership_type',
    'marital_status', 'is_resident', 'receives_primary_income_at_bank',
]
TENURE_COLS = [
    'months_since_first_relationship', 'months_since_last_relationship_start',
    'acquisition_channel',
]
HOLDINGS_COLS = [
    'n_products_total', 'n_products_active', 'n_products_closed_last_12m',
    'n_domains', 'n_categories',
    'has_checking', 'has_savings', 'has_credit_card', 'has_loan', 'has_mortgage', 'has_investment',
    'product_diversity_score',
]
BALANCE_COLS = [
    'total_balance_avg', 'total_balance_end', 'total_balance_start',
    'balance_trend_slope', 'balance_volatility', 'balance_min', 'balance_max',
    'balance_end_to_start_ratio',
]
TX_COLS = [
    'tx_count_total', 'tx_count_last90d', 'tx_count_last30d', 'tx_count_ratio_30d_180d',
    'tx_amount_total', 'tx_amount_last90d', 'tx_amount_avg_per_tx', 'tx_amount_median',
    'tx_inflow_total', 'tx_outflow_total', 'tx_net_flow',
    'tx_count_trend_slope', 'tx_amount_trend_slope', 'tx_month_count_ratio',
    'days_since_last_tx',
]
CHANNEL_COLS = [
    'channel_diversity',
    'share_by_channel_atm', 'share_by_channel_online', 'share_by_channel_branch',
    'share_by_channel_mobile', 'share_by_channel_pos',
    'mobile_adoption_last90d',
]
CATEGORY_COLS = [f'spend_share_{e["sanitized"]}' for e in whitelist] + ['category_diversity']
INTL_COLS = ['foreign_tx_share', 'counterparty_diversity']

FEATURE_COLS = DEMO_COLS + TENURE_COLS + HOLDINGS_COLS + BALANCE_COLS + TX_COLS + CHANNEL_COLS + CATEGORY_COLS + INTL_COLS

missing = [c for c in FEATURE_COLS if c not in features.columns]
if missing:
    print(f'WARNING: missing columns: {missing}')
else:
    print(f'All {len(FEATURE_COLS)} feature columns present.')

All 82 feature columns present.


In [29]:
features_ordered = features[['IDENTIFIKATOR_KLIJENTA'] + FEATURE_COLS].copy()

# Replace any inf/-inf with NaN
num_cols = features_ordered.select_dtypes(include=[np.number]).columns
features_ordered[num_cols] = features_ordered[num_cols].replace([np.inf, -np.inf], np.nan)

print(f'Final feature matrix: {features_ordered.shape}')
features_ordered.dtypes.value_counts()

Final feature matrix: (4614, 83)


float64    51
int64      19
object     13
Name: count, dtype: int64

## 12. Save outputs

In [31]:
out_parquet   = PROC   / 'features_churn.parquet'
out_cols      = MODELS / 'churn_feature_columns.json'
out_whitelist = MODELS / 'churn_category_whitelist.json'

features_ordered.to_parquet(out_parquet, index=False)
print(f'Saved: {out_parquet}')

with open(out_cols, 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)
print(f'Saved: {out_cols}  ({len(FEATURE_COLS)} columns)')

with open(out_whitelist, 'w') as f:
    json.dump(whitelist, f, indent=2, ensure_ascii=False)
print(f'Saved: {out_whitelist}  ({len(whitelist)} categories)')

Saved: /Users/Max/Desktop/Fintech-hackathon/data/processed/features_churn.parquet
Saved: /Users/Max/Desktop/Fintech-hackathon/data/models/churn_feature_columns.json  (82 columns)
Saved: /Users/Max/Desktop/Fintech-hackathon/data/models/churn_category_whitelist.json  (15 categories)


## 13. Acceptance Criteria Checks

In [32]:
print('=' * 60)
print('ACCEPTANCE CRITERIA CHECKS')
print('=' * 60)

# Reload from disk to validate persistence
feat_check     = pd.read_parquet(out_parquet)
with open(out_cols)      as f: feat_col_list    = json.load(f)
with open(out_whitelist) as f: whitelist_check  = json.load(f)

expected_n = len(train_ids) + len(test_ids) + len(holdout_ids)

# 1. Row count
assert len(feat_check) == expected_n, f'Row count: {len(feat_check)} != {expected_n}'
assert len(feat_check) == len(clients), f'Row count != eligible universe'
print(f'[PASS] 1. Row count: {len(feat_check):,} == {expected_n:,}')

# 2. Client ID uniqueness and completeness
assert feat_check['IDENTIFIKATOR_KLIJENTA'].nunique() == expected_n, 'Duplicate client IDs!'
assert set(feat_check['IDENTIFIKATOR_KLIJENTA']) == all_eligible_ids, 'Client ID set mismatch!'
print(f'[PASS] 2. IDENTIFIKATOR_KLIJENTA: {feat_check["IDENTIFIKATOR_KLIJENTA"].nunique():,} unique, matches eligible set')

# 3. Column list order matches parquet
parquet_fcols = [c for c in feat_check.columns if c != 'IDENTIFIKATOR_KLIJENTA']
assert feat_col_list == parquet_fcols, 'Column list order mismatch!'
print(f'[PASS] 3. churn_feature_columns.json matches parquet column order ({len(feat_col_list)} columns)')

# 4. Whitelist has exactly 15 entries
assert len(whitelist_check) == 15, f'Whitelist has {len(whitelist_check)} entries, expected 15'
print(f'[PASS] 4. churn_category_whitelist.json has exactly 15 entries')

# 5. spend_share_* sum <= 1.0 per row
sc = [c for c in feat_check.columns if c.startswith('spend_share_')]
max_sum = feat_check[sc].fillna(0).sum(axis=1).max()
assert max_sum <= 1.0 + 1e-6, f'spend_share sum > 1: {max_sum:.6f}'
print(f'[PASS] 5. spend_share_* sum <= 1.0 per row (max={max_sum:.4f})')

# 6. No tx timestamp > feature_window_end
assert max_tx_date <= feature_window_end, f'LEAKAGE: max_tx_date {max_tx_date}'
print(f'[PASS] 6. Max tx date {max_tx_date} <= feature_window_end')

# 7. No +inf or -inf
nc = feat_check.select_dtypes(include=[np.number]).columns
assert not np.isinf(feat_check[nc].values).any(), 'Found inf values!'
print(f'[PASS] 7. No +inf/-inf in any numeric column')

# 8. Column count sanity check: >= 50 and <= 90.
# The spec's 8 groups as written yield 82 columns:
# 19 demo + 3 tenure + 12 holdings + 8 balance + 15 tx + 7 channel
# + 16 category (15 spend_share + 1 diversity) + 2 international = 82.
# Upper bound 90 catches accidental explosions without rejecting a correct run.
n_feat = len(feat_col_list)
assert 50 <= n_feat <= 90, f'Column count {n_feat} not in [50, 90]'
print(f'[PASS] 8. Feature column count: {n_feat} (in range [50, 90])')
print()
print('ALL CHECKS PASSED.')

ACCEPTANCE CRITERIA CHECKS
[PASS] 1. Row count: 4,614 == 4,614
[PASS] 2. IDENTIFIKATOR_KLIJENTA: 4,614 unique, matches eligible set
[PASS] 3. churn_feature_columns.json matches parquet column order (82 columns)
[PASS] 4. churn_category_whitelist.json has exactly 15 entries
[PASS] 5. spend_share_* sum <= 1.0 per row (max=1.0000)
[PASS] 6. Max tx date 2025-12-15 23:51:58+00:00 <= feature_window_end
[PASS] 7. No +inf/-inf in any numeric column
[PASS] 8. Feature column count: 82 (in range [50, 90])

ALL CHECKS PASSED.


In [33]:
print('=' * 60)
print('FEATURE ENGINEERING SUMMARY')
print('=' * 60)
print(f'Total rows         : {len(feat_check):,}')
print(f'Total feature cols : {len(feat_col_list)}')
print()

groups = [
    ('Demographics',     DEMO_COLS),
    ('Tenure',           TENURE_COLS),
    ('Product holdings', HOLDINGS_COLS),
    ('Balance',          BALANCE_COLS),
    ('Transactions',     TX_COLS),
    ('Channels',         CHANNEL_COLS),
    ('Category spend',   CATEGORY_COLS),
    ('International',    INTL_COLS),
]

print(f'{"Group":<20} {"Cols":>5} {"Rows w/ NaN":>12} {"NaN %":>8}')
print('-' * 50)
for grp_name, grp_cols in groups:
    nan_rows = feat_check[grp_cols].isna().any(axis=1).sum()
    nan_pct  = nan_rows / len(feat_check) * 100
    print(f'{grp_name:<20} {len(grp_cols):>5} {nan_rows:>12,} {nan_pct:>7.1f}%')

print()
print('Category whitelist (top 15 by outflow):')
for i, entry in enumerate(whitelist_check, 1):
    print(f'  {i:2d}. {entry["original"]}  ->  spend_share_{entry["sanitized"]}')

FEATURE ENGINEERING SUMMARY
Total rows         : 4,614
Total feature cols : 82

Group                 Cols  Rows w/ NaN    NaN %
--------------------------------------------------
Demographics            19        4,614   100.0%
Tenure                   3        4,614   100.0%
Product holdings        12            0     0.0%
Balance                  8        2,018    43.7%
Transactions            15          157     3.4%
Channels                 7          273     5.9%
Category spend          16          814    17.6%
International            2            0     0.0%

Category whitelist (top 15 by outflow):
   1. Service providers  ->  spend_share_SERVICE_PROVIDERS
   2. Retail outlets  ->  spend_share_RETAIL_OUTLETS
   3. JAVNA UPRAVA I OBRANA; OBVEZNO SOCIJALNO OSIGURANJE  ->  spend_share_JAVNA_UPRAVA_I_OBRANA_OBVEZNO_SOCIJALNO_OSIGURANJE
   4. FINANCIJSKE DJELATNOSTI I DJELATNOSTI OSIGURANJA  ->  spend_share_FINANCIJSKE_DJELATNOSTI_I_DJELATNOSTI_OSIGURANJA
   5. GRAÐEVINARSTVO  ->  sp